# Testing retrieval on FiQA and LongBench v2

Kevin's external benchmark experiments. The same retrieval families are evaluated using the labels each dataset supplies.

**FiQA:** did retrieval find relevant financial answers? **LongBench v2:** did an answer model choose the correct option after reading retrieved text?

The saved results below are from completed local runs. The Legal RAG evidence-span labels are not reused as labels for these datasets.

## 1. Load the saved experiment

Open this notebook inside the external_rag_benchmarks folder. Viewing saved results needs no GPU. Model reruns are controlled by flags below.

In [1]:
from pathlib import Path
import csv, json, html, sys, subprocess
from IPython.display import display, HTML
ROOT = Path.cwd()
if not (ROOT/'retrieval.py').exists():
    ROOT = ROOT/'external_rag_benchmarks'
assert (ROOT/'retrieval.py').exists(), 'Open the notebook in the benchmark folder or its parent.'
sys.path.insert(0,str(ROOT))
from retrieval import read
def rows(path):
    with (ROOT/path).open(encoding='utf-8') as f:
        return list(csv.DictReader(f))
def show(records, columns=None):
    records = list(records)
    if not records:
        print('No rows available.')
        return
    columns = columns or list(records[0])
    head = ''.join('<th>'+html.escape(str(c))+'</th>' for c in columns)
    body = ''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in columns)+'</tr>' for r in records)
    display(HTML('<table><tr>'+head+'</tr>'+body+'</table>'))
manifest = read(ROOT/'data/selection_manifest.json')
config = read(ROOT/'config.json')
budget_config = read(ROOT/'longbench_budget_config.json')
show([{'Dataset':'FiQA','Corpus entries':manifest['fiqa']['corpus'],'Development':50,'Test':100},
      {'Dataset':'LongBench short QA pilot','Corpus entries':'Each item has its own context','Development':6,'Test':14}])


Dataset,Corpus entries,Development,Test
FiQA,57638,50,100
LongBench short QA pilot,Each item has its own context,6,14


## 2. Shared retrieval methods

BM25 matches words. Qwen3-Embedding-0.6B produces normalized semantic vectors, with a fixed 512-token input window. Hybrid combines the rankings using reciprocal rank fusion, which avoids adding incomparable raw scores.

The MiniLM tokenizer only counts LongBench chunk lengths and context budgets. No MiniLM embedding model is run. FiQA indexes whole original corpus entries; LongBench splits its supplied contexts into 256-token chunks with overlap.

In [2]:
show([{'Setting':k,'Value':config[k]} for k in ['embedding_model','embedding_window','bm25_k1','bm25_b','rrf_constant','candidate_depth','longbench_chunk_tokens','longbench_overlap','reader_model']]
     + [{'Setting':'Revised retrieval depth','Value':budget_config['top_k']},
        {'Setting':'Revised body budgets','Value':'2048, 8192, all top-40 text'}])


Setting,Value
embedding_model,Qwen/Qwen3-Embedding-0.6B
embedding_window,512
bm25_k1,1.5
bm25_b,0.75
rrf_constant,60
candidate_depth,100
longbench_chunk_tokens,256
longbench_overlap,32
reader_model,Qwen/Qwen2.5-1.5B-Instruct
Revised retrieval depth,40


## 3. FiQA: relevant documents rather than exact spans

We search all 57,638 corpus entries. Fifty development questions select the fusion weight; the fixed 100-question test sample reports performance. The selected test sample has 71 questions with multiple relevant documents.

Hit@10 asks whether any relevant document was found. Recall@10 asks what fraction of relevant documents was found. nDCG and MRR reward better ranking. Multiple relevant answers may be alternatives, so this is not a complete-evidence span test.

In [3]:
fiqa = rows('results/fiqa/comparison.csv')
show([r for r in fiqa if r['split']=='test' and r['k']=='10'], ['method','queries','hit','recall','precision','ndcg','mrr','all_judged_relevant_retrieved'])


method,queries,hit,recall,precision,ndcg,mrr,all_judged_relevant_retrieved
BM25,100,0.41,0.2530930735930736,0.061000000000000006,0.2173452856215793,0.2865515873015873,0.15
Qwen dense,100,0.81,0.5600108225108226,0.13699999999999998,0.49203900878458384,0.5889801587301587,0.35
Hybrid equal,100,0.67,0.4692251082251082,0.113,0.3769325014280724,0.4449404761904762,0.29
Hybrid selected,100,0.75,0.5371731601731602,0.129,0.44319070836540775,0.5138174603174603,0.34


## 4. Development-only fusion selection

The BM25 weight ranges from 0 to 1. Zero means pure dense retrieval; one means pure BM25. We maximize development nDCG@10, then Recall@10, with a tie preference toward equal weighting. The test scores do not select the weight.

In [4]:
show(rows('results/fiqa/tuning_dev.csv'))
print('Frozen selection:',read(ROOT/'results/fiqa/selection.json')['bm25_weight'])
print('All test top-k settings:')
show([r for r in fiqa if r['split']=='test'],['method','k','hit','recall','ndcg'])


Frozen selection: 0.25
All test top-k settings:


bm25_weight,ndcg10,recall10
0,0.3258649114035769,0.38504273504273506
0.25,0.34403766308422895,0.3817094017094017
0.5,0.3209762502890424,0.3712820512820512
0.75,0.27816417590587195,0.3682051282051282
1,0.18200111696378865,0.24166666666666664


method,k,hit,recall,ndcg
BM25,1,0.24,0.11326623376623374,0.24
BM25,5,0.34,0.19167965367965364,0.19687254006731764
BM25,10,0.41,0.2530930735930736,0.2173452856215793
Qwen dense,1,0.5,0.27076623376623377,0.5
Qwen dense,5,0.7,0.4540887445887446,0.458919232242826
Qwen dense,10,0.81,0.5600108225108226,0.49203900878458384
Hybrid equal,1,0.35,0.16309956709956708,0.35
Hybrid equal,5,0.57,0.37663636363636366,0.34640165732151845
Hybrid equal,10,0.67,0.4692251082251082,0.3769325014280724
Hybrid selected,1,0.41,0.20409956709956706,0.41


## 5. LongBench v2: answering from retrieved context

This pilot selects twenty short single- and multi-document QA items by a fixed hash. Each question searches its own supplied context. The question and options enter retrieval; the correct answer letter does not.

For each method we hold the same top-forty ranking fixed and deliver either 2,048 tokens, 8,192 tokens or all forty complete chunks. The final condition removes the additional body cutoff; it does not give the reader the entire original source. The same Qwen2.5-1.5B-Instruct reader generates an option letter using greedy decoding and the same prompt instructions. A no-context control is also included.

The six development and fourteen test items are the previously inspected project sample. This budget follow-up is exploratory. Each full prompt is checked with the reader's tokenizer against its supported input capacity; no silent truncation is allowed.

In [5]:
longbench = rows('results/longbench_budget_sweep/comparison.csv')
timings = {(r['split'],r['method'],r['budget']):r for r in rows('results/longbench_budget_sweep/timing_summary.csv')}
show([{'Method':r['method'],'Budget':r['budget'],'Correct / 14':r['correct']+'/14',
       'Accuracy':f"{float(r['accuracy']):.1%}",'Mean body tokens':round(float(r['mean_returned_tokens']),1),
       'Median generation seconds':round(float(timings[(r['split'],r['method'],r['budget'])]['median_generation_seconds']),3)}
      for r in longbench if r['split']=='test'])
print('Changes relative to the same retriever at 2,048 tokens:')
show([r for r in rows('results/longbench_budget_sweep/paired_changes.csv') if r['split']=='test'])


Changes relative to the same retriever at 2,048 tokens:


Method,Budget,Correct / 14,Accuracy,Mean body tokens,Median generation seconds
No context,none,2/14,14.3%,0.0,0.117
BM25,2048,4/14,28.6%,2048.0,0.36
BM25,8192,4/14,28.6%,8192.0,1.633
BM25,all_retrieved,5/14,35.7%,10239.1,2.164
Qwen dense,2048,3/14,21.4%,2048.0,0.36
Qwen dense,8192,3/14,21.4%,8192.0,1.655
Qwen dense,all_retrieved,3/14,21.4%,10221.2,2.199
Hybrid equal,2048,3/14,21.4%,2048.0,0.349
Hybrid equal,8192,3/14,21.4%,8192.0,1.63
Hybrid equal,all_retrieved,3/14,21.4%,10236.9,2.159


split,method,budget,questions,wins_vs_2048,losses_vs_2048,ties
test,BM25,8192,14,1,1,12
test,BM25,all_retrieved,14,2,1,11
test,Qwen dense,8192,14,0,0,14
test,Qwen dense,all_retrieved,14,0,0,14
test,Hybrid equal,8192,14,2,2,10
test,Hybrid equal,all_retrieved,14,1,1,12


## 6. Inspect a real prediction

A correct letter is an end-to-end outcome; it does not tell us which evidence was used. Differences may come from retrieval, missing context, model knowledge or reader errors. With fourteen test items, one question changes accuracy by about 7.1 percentage points.

BM25's observed correct count changes from 4 to 4 to 5 as the budget increases. Dense and hybrid each stay at 3, although hybrid's individual wins and losses offset each other. This pilot does not show a consistent gain from adding more retrieved text.

In [6]:
answers = read(ROOT/'results/longbench_budget_sweep/answers.json')
test_answers = [a for a in answers if a['split']=='test']
ids = list(dict.fromkeys(a['id'] for a in test_answers))
example_id = next((i for i in ids if len({a['correct'] for a in test_answers if a['id']==i})>1),ids[0])
import gzip
with gzip.open(ROOT/'data/longbench_pilot.jsonl.gz','rt',encoding='utf-8') as f:
    example = next(q for q in map(json.loads,f) if q['_id']==example_id)
print('Question:',example['question'])
for letter in 'ABCD':
    print(letter+'.',example['choice_'+letter])
show([a for a in test_answers if a['id']==example_id],['id','domain','method','budget','raw_response','correct_answer','correct'])


Question: Which one is noted in all the five passages?
A. the democracy
B. the education of the young people who are allways considered as the sun of the nation
C. the cooperation between various people
D. the bright future of China


id,domain,method,budget,raw_response,correct_answer,correct
66fbab85bb02136c067c81dc,Multi-Document QA,No context,none,B,C,0
66fbab85bb02136c067c81dc,Multi-Document QA,BM25,2048,B,C,0
66fbab85bb02136c067c81dc,Multi-Document QA,BM25,8192,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,BM25,all_retrieved,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,Qwen dense,2048,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,Qwen dense,8192,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,Qwen dense,all_retrieved,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,Hybrid equal,2048,B,C,0
66fbab85bb02136c067c81dc,Multi-Document QA,Hybrid equal,8192,C,C,1
66fbab85bb02136c067c81dc,Multi-Document QA,Hybrid equal,all_retrieved,C,C,1


## 7. Compute and verification

GPU corpus encoding, CPU similarity search and local answer generation are separate stages. The timing files describe component costs from the observed run. No paid inference API is used.

One BM25 8,192-token generation took about 299 seconds, an unexplained runtime outlier. The table above uses median generation time. Raw times, mean and maximum remain available in timing_summary.csv; this was not a repeated latency benchmark.

The audit independently recomputes retrieval scores and answer accuracy, verifies dataset hashes and delivered source slices, and checks development-based fusion selection. Neither benchmark supplies the exact supporting spans needed for the Legal RAG span metrics.

In [7]:
fm = read(ROOT/'results/fiqa/run_manifest.json')
show([{'GPU':fm['gpu'],'FiQA encoding seconds':round(fm['profile']['encoding_seconds'],2),
       'Dense truncation rate':fm['profile']['input_truncation_rate'],'Corpus entries':fm['corpus_count']}])
show([read(ROOT/'results/audit.json')])
show([read(ROOT/'results/longbench_budget_sweep/audit.json')])


GPU,FiQA encoding seconds,Dense truncation rate,Corpus entries
NVIDIA GeForce RTX 5070 Laptop GPU,797.68,0.04039002047260488,57638


fiqa_query_metric_rows_checked,fiqa_summary_rows_checked,longbench_predictions_checked,longbench_summary_rows_checked,software_tests_passed,source_hashes_and_context_slices,frozen_dev_selection,limits
1800,24,80,8,19,passed,passed,"Audit checks arithmetic and protocol, not correctness of publisher relevance judgments or model reasoning"


contexts_checked,answers_checked,summary_rows_checked,same_top40_ranking_across_budgets,all_uncapped_conditions_include_all40,reader_prompt_truncations,maximum_native_prompt_tokens,original_2048_contexts_identical,prediction_changes_vs_original_baseline,status,interpretation
180,200,20,True,True,0,14047,True,0,passed,Exploratory follow-up on previously inspected questions


## 8. Optional reproduction

The runs are already complete. Enable the flags only to reproduce them in an environment with the requirements installed. Embedding blocks are cached and completed answer records resume. Read README.md before changing settings. Run a fresh experiment directory when changing the protocol.

In [8]:
RUN_FIQA = False
RUN_LONGBENCH = False
if RUN_FIQA:
    subprocess.run([sys.executable,'run_fiqa.py'],cwd=ROOT,check=True)
if RUN_LONGBENCH:
    subprocess.run([sys.executable,'run_longbench_budgets.py','contexts'],cwd=ROOT,check=True)
    subprocess.run([sys.executable,'run_longbench_budgets.py','answer'],cwd=ROOT,check=True)
